In [1]:
import os

for root, dirs, files in os.walk("/opt/spark/python"):
    for f in files:
        if "py4j" in f:
            print(os.path.join(root, f))

/opt/spark/python/lib/py4j-0.10.9.9-src.zip


In [2]:
import sys
import glob

sys.path.insert(0, "/opt/spark/python")

py4j_path = glob.glob("/opt/spark/python/lib/py4j*.zip")[0]
sys.path.insert(0, py4j_path)

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    trim,
    when,
    regexp_replace,
    from_unixtime
)
from pyspark.sql import functions as F

spark = (SparkSession.builder
    .appName("Amazon Video Games")
    .master("local[4]")                          # bukan local[*]
    .config("spark.driver.memory", "6g")         # sesuaikan: ±50-60% memori Docker
    .config("spark.driver.maxResultSize", "1g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.hadoop.dfs.replication", "1") # HDFS kamu hanya punya 1 datanode
    .getOrCreate())

print(spark.sparkContext.getConf().get("spark.driver.memory"))
print(spark._jvm.Runtime.getRuntime().maxMemory() / 1e9, "GB heap")
print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 09:52:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


6g
6.442450944 GB heap
Spark version: 4.2.0


In [3]:
reviews = spark.read.json(
    "hdfs://namenode:9000/data/amazon_reviews/reviews.json"
)
metadata = spark.read.json(
    "hdfs://namenode:9000/data/amazon_reviews/metadata.json"
)

In [4]:
print("Reviews :", reviews.count())
print("Metadata:", metadata.count())

26/11/04 09:52:51 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Reviews : 2565349
Metadata: 84819


In [6]:
reviews.printSchema()
metadata.printSchema()

root
 |-- asin: string (nullable = true)
 |-- image: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- overall: double (nullable = true)
 |-- reviewText: string (nullable = true)
 |-- reviewTime: string (nullable = true)
 |-- reviewerID: string (nullable = true)
 |-- reviewerName: string (nullable = true)
 |-- style: struct (nullable = true)
 |    |-- Color Name:: string (nullable = true)
 |    |-- Color:: string (nullable = true)
 |    |-- Configuration:: string (nullable = true)
 |    |-- Content:: string (nullable = true)
 |    |-- Denomination:: string (nullable = true)
 |    |-- Edition:: string (nullable = true)
 |    |-- Format:: string (nullable = true)
 |    |-- Item Package Quantity:: string (nullable = true)
 |    |-- Length:: string (nullable = true)
 |    |-- Offer Type:: string (nullable = true)
 |    |-- Package Quantity:: string (nullable = true)
 |    |-- Package Type:: string (nullable = true)
 |    |-- Pattern:: string (nullable = true)
 | 

Transform Reviews

In [5]:
total_rows = reviews.count()
unique_rows = reviews.dropDuplicates().count()

duplicate_rows = total_rows - unique_rows

print(f"Total rows     : {total_rows:,}")
print(f"Unique rows    : {unique_rows:,}")
print(f"Duplicate rows : {duplicate_rows:,}")

[Stage 13:============================>                            (8 + 4) / 16]

Total rows     : 2,565,349
Unique rows    : 2,491,137
Duplicate rows : 74,212


In [6]:
reviews = reviews.dropDuplicates()

In [7]:
reviews = reviews.dropDuplicates()

print(reviews.count())

[Stage 10:==========================================>             (12 + 4) / 16]

2491137


Udah sama berarti aman Duplicate Rows = 74,212

In [8]:
reviews.filter(F.col("asin").isNull()).count()


0

In [11]:
reviews.filter(
    F.col("asin").isNull() |
    (F.trim(F.col("asin")) == "") |
    (F.trim(F.col("asin")) == " ")
).count()

0

In [13]:
reviews.filter(
    F.col("reviewerID").isNull() |
    (F.trim(F.col("reviewerID")) == "") |
    (F.trim(F.col("reviewerID")) == " ")
).count()

0

Walaupun 0 buat jaga jaga coba aja

In [7]:
reviews = reviews.filter(col("asin").isNotNull())
reviews = reviews.filter(col("reviewerID").isNotNull())
reviews = reviews.fillna(
    {
        "reviewText": ""
    }
)

In [8]:
reviews = reviews.filter(col("asin").isNotNull())
print(reviews.count())

[Stage 19:=================================================>      (14 + 2) / 16]

2491137


In [11]:
reviews = reviews.filter(col("reviewerID").isNotNull())
print(reviews.count())

[Stage 25:==========================================>             (12 + 4) / 16]

2491137


In [6]:
reviews.filter(
    F.col("reviewText").isNull() |
    (F.trim(F.col("reviewText")) == "") |
    (F.trim(F.col("reviewText")) == " ")
).count()

1698

In [9]:
reviews = reviews.fillna(
    {
        "reviewText": ""
    }
)

In [7]:
reviews.filter(
    (F.trim(F.col("reviewText")) == "")
).count()

1698

In [10]:
reviews = reviews.withColumn(
    "review_datetime",
    from_unixtime("unixReviewTime")
)

In [11]:
reviews = reviews.withColumn(
    "vote",
    regexp_replace("vote", ",", "")
)

In [12]:
reviews = reviews.withColumn(
    "vote",
    col("vote").cast("int")
)

In [13]:
from pyspark.sql.functions import col, sum

reviews.select(
    sum(col("vote").isNull().cast("int")).alias("null_vote")
).show()

[Stage 25:==========================================>             (12 + 4) / 16]

+---------+
|null_vote|
+---------+
|  2067511|
+---------+



In [14]:
reviews.groupBy("vote") \
       .count() \
       .orderBy(col("count").desc()) \
       .show(20)

[Stage 31:==========================================>             (12 + 4) / 16]

+----+-------+
|vote|  count|
+----+-------+
|NULL|2067511|
|   2| 134159|
|   3|  74009|
|   4|  45688|
|   5|  31453|
|   6|  22660|
|   7|  17078|
|   8|  13293|
|   9|  10532|
|  10|   8776|
|  11|   7245|
|  12|   5922|
|  13|   4877|
|  14|   4335|
|  15|   3636|
|  16|   3294|
|  17|   2714|
|  18|   2442|
|  19|   2188|
|  20|   1898|
+----+-------+
only showing top 20 rows


In [15]:
reviews.filter(
    col("vote") == 1
).count()

0

In [15]:
reviews = reviews.fillna(
    {"vote": 0}
)

In [16]:
reviews.show(10)

[Stage 35:===================================================>    (12 + 1) / 13]

+----------+-----+-------+--------------------+-----------+--------------+---------------+-----+--------------------+--------------+--------+----+-------------------+
|      asin|image|overall|          reviewText| reviewTime|    reviewerID|   reviewerName|style|             summary|unixReviewTime|verified|vote|    review_datetime|
+----------+-----+-------+--------------------+-----------+--------------+---------------+-----+--------------------+--------------+--------+----+-------------------+
|0439381673| NULL|    5.0|  Great game to play|12 26, 2016|A3S1M68MUHPJ11|      w. Rieder| NULL|          Five Stars|    1482710400|    true|   0|2016-12-26 00:00:00|
|0439381673| NULL|    1.0|Does not work on ...| 03 5, 2015| AR9984T786QGT|      marybird9| NULL|            One Star|    1425513600|    true|   0|2015-03-05 00:00:00|
|0439381673| NULL|    1.0|My son got this f...|12 18, 2014|A1WEOXIAQEBWZR|            MNY| NULL|Didn't work on ou...|    1418860800|    true|   0|2014-12-18 00:00:00

In [17]:
reviews = reviews.drop(
    "image", # banyak yang null terus ga guna
    "style", # ga kepake nanti
    "reviewerName", # nama ga boleh (Data Privacy)
    "reviewTime",
    "unixReviewTime"
)

In [18]:
reviews.show(5)

[Stage 38:===================================================>    (12 + 1) / 13]

+----------+-------+--------------------+--------------+--------------------+--------+----+-------------------+
|      asin|overall|          reviewText|    reviewerID|             summary|verified|vote|    review_datetime|
+----------+-------+--------------------+--------------+--------------------+--------+----+-------------------+
|0439381673|    5.0|  Great game to play|A3S1M68MUHPJ11|          Five Stars|    true|   0|2016-12-26 00:00:00|
|0439381673|    1.0|Does not work on ...| AR9984T786QGT|            One Star|    true|   0|2015-03-05 00:00:00|
|0439381673|    1.0|My son got this f...|A1WEOXIAQEBWZR|Didn't work on ou...|    true|   0|2014-12-18 00:00:00|
|0700026657|    5.0|excellent service...|A2LK3J4OAGFJ7L|          Five Stars|    true|   0|2015-05-15 00:00:00|
|0700026657|    4.0|i liked a lot som...|A11V6ZJ2FVQY1D|           Anno 2070|    true|   0|2014-11-13 00:00:00|
+----------+-------+--------------------+--------------+--------------------+--------+----+-------------

Transform Metadata

In [19]:
total_rows = metadata.count()
unique_rows = metadata.dropDuplicates().count()

duplicate_rows = total_rows - unique_rows

print(f"Total rows     : {total_rows:,}")
print(f"Unique rows    : {unique_rows:,}")
print(f"Duplicate rows : {duplicate_rows:,}")

[Stage 44:>                                                         (0 + 4) / 4]

Total rows     : 84,819
Unique rows    : 71,911
Duplicate rows : 12,908


In [20]:
metadata = metadata.dropDuplicates()

In [17]:
metadata = metadata.dropDuplicates()

print(metadata.count())

[Stage 26:=======>                                                (2 + 14) / 16]

71911


In [21]:
metadata = metadata.withColumn(
    "brand",
    regexp_replace(
        "brand",
        r"by\s*",
        ""
    )
)

In [22]:
metadata = metadata.withColumn(
    "brand",
    regexp_replace(
        "brand",
        r"\n",
        ""
    )
)

In [23]:
metadata = metadata.withColumn(
    "brand",
    trim(col("brand"))
)

In [24]:
metadata = metadata.fillna(
    {
        "brand": "Unknown"
    }
)

In [22]:
metadata.filter(F.col("brand").isNull()).count()

0

In [18]:
metadata.filter(F.col("brand") == "").count()

3617

In [25]:
metadata = metadata.withColumn(
    "brand",
    F.when(
        F.col("brand").isNull() | (F.trim(F.col("brand")) == ""),
        "Unknown"
    ).otherwise(F.col("brand"))
)

In [26]:
from pyspark.sql import functions as F

for col_name in ["title", "main_cat"]:
    metadata = metadata.withColumn(
        col_name,
        F.when(
            F.col(col_name).isNull() | (F.trim(F.col(col_name)) == ""),
            "Unknown"
        ).otherwise(F.col(col_name))
    )

In [27]:
metadata.groupBy("brand") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(10, truncate=False)

[Stage 50:=============================>                            (2 + 2) / 4]

+---------------+-----+
|brand          |count|
+---------------+-----+
|Unknown        |3945 |
|Nintendo       |2704 |
|Electronic Arts|2559 |
|Activision     |2328 |
|Sony           |1727 |
|Ubisoft        |1608 |
|Sega           |1536 |
|Microsoft      |1191 |
|Atari          |872  |
|THQ            |858  |
+---------------+-----+
only showing top 10 rows


In [28]:
total = metadata.count()

unknown = (
    metadata.filter(F.col("brand") == "Unknown")
    .count()
)

print(f"Unknown percentage: {unknown/total*100:.2f}%")

Unknown percentage: 5.49%


In [29]:
metadata.show(5)

[Stage 68:==============>                                           (1 + 3) / 4]

+------------+--------------------+----------+------------+--------------------+----+--------------------+-------+--------------------+---+--------------------+--------------------+----------------+--------------------+--------------------+------------+-----+-----+--------------------+
|    also_buy|           also_view|      asin|       brand|            category|date|         description|details|             feature|fit|            imageURL|     imageURLHighRes|        main_cat|               price|                rank|similar_item|tech1|tech2|               title|
+------------+--------------------+----------+------------+--------------------+----+--------------------+-------+--------------------+---+--------------------+--------------------+----------------+--------------------+--------------------+------------+-----+-----+--------------------+
|          []|[B00005V8TB, B008...|0439542804|        LEGO|[Video Games, PC,...|    |[There is so much...|   NULL|[The action adven...|   |

In [30]:
# Dari EDA banyak yang kosong jadi mending di drop aja 
metadata = metadata.drop(
    "fit",
    "tech1",
    "tech2",
    "date",
    "similar_item",
    "imageURL",
    "imageURLHighRes",
)

In [31]:
metadata.show(5)

[Stage 71:=============================>                            (2 + 2) / 4]

+------------+--------------------+----------+------------+--------------------+--------------------+-------+--------------------+----------------+--------------------+--------------------+--------------------+
|    also_buy|           also_view|      asin|       brand|            category|         description|details|             feature|        main_cat|               price|                rank|               title|
+------------+--------------------+----------+------------+--------------------+--------------------+-------+--------------------+----------------+--------------------+--------------------+--------------------+
|          []|[B00005V8TB, B008...|0439542804|        LEGO|[Video Games, PC,...|[There is so much...|   NULL|[The action adven...|Toys &amp; Games|                    |[">#930,608 in To...|  LEGO Island 2 - PC|
|[B079X6BSYT]|                  []|0977070670|     Unknown|                  []|[One day, a beaut...|   NULL|                  []|     Video Games|         

In [32]:
# price masih kurang bagus

metadata.groupBy("price") \
        .count() \
        .orderBy("count", ascending=False) \
        .show(20, truncate=False)

[Stage 74:>                                                         (0 + 4) / 4]

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [33]:
from pyspark.sql.functions import col, regexp_extract, when

metadata = metadata.withColumn(
    "price_clean",
    regexp_extract(
        col("price"),
        r"\$([0-9]+\.[0-9]{2})",
        1
    )
)

metadata = metadata.withColumn(
    "price_clean",
    when(
        col("price_clean") == "",
        None
    ).otherwise(col("price_clean"))
)

metadata = metadata.withColumn(
    "price_clean",
    col("price_clean").cast("double")
)

In [42]:
metadata.show(5)

+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+--------------------+--------------------+--------------------+-----------+
|            also_buy|           also_view|      asin|               brand|            category|         description|             details|             feature|   main_cat|               price|                rank|               title|price_clean|
+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+--------------------+--------------------+--------------------+-----------+
|[B0028ZJ4ZW, B003...|[B01D47DUD4, B005...|6050036071|            Logitech|[Video Games, Pla...|[Official Rock Ba...|                NULL|[High-performance...|Video Games|\n\t\t           ...|[">#3,867 in Vide...|USB Microphone fo...|       NULL|
|           

In [34]:
from pyspark.sql.functions import col, count, when

metadata.select(
    count("*").alias("total_product"),
    count(when(col("price_clean").isNotNull(), True)).alias("valid_price"),
    count(when(col("price_clean").isNull(), True)).alias("null_price")
).show()

[Stage 80:==============>                                           (1 + 3) / 4]

+-------------+-----------+----------+
|total_product|valid_price|null_price|
+-------------+-----------+----------+
|        71911|      11905|     60006|
+-------------+-----------+----------+



In [35]:
from pyspark.sql.functions import min, max, avg
metadata.select(
    min("price_clean").alias("min_price"),
    max("price_clean").alias("max_price"),
    avg("price_clean").alias("avg_price")
).show()

[Stage 86:=============================>                            (2 + 2) / 4]

+---------+---------+-----------------+
|min_price|max_price|        avg_price|
+---------+---------+-----------------+
|      0.0|   999.99|37.75001175976505|
+---------+---------+-----------------+



In [36]:
metadata.filter(
    col("price_clean") < 0
).count()

0

In [37]:
metadata.select(
    "price",
    "price_clean"
).where(
    col("price_clean").isNotNull()
).show(20, truncate=False)

+-------+-----------+
|price  |price_clean|
+-------+-----------+
|$20.99 |20.99      |
|$51.94 |51.94      |
|$43.51 |43.51      |
|$499.95|499.95     |
|$16.54 |16.54      |
|$349.99|349.99     |
|$1.99  |1.99       |
|$36.57 |36.57      |
|$64.99 |64.99      |
|$94.60 |94.6       |
|$75.45 |75.45      |
|$19.99 |19.99      |
|$49.99 |49.99      |
|$49.99 |49.99      |
|$42.64 |42.64      |
|$46.95 |46.95      |
|$129.51|129.51     |
|$17.34 |17.34      |
|$25.00 |25.0       |
|$36.52 |36.52      |
+-------+-----------+
only showing top 20 rows


In [38]:
metadata.filter(
    col("price_clean") == 0
).select(
    "asin",
    "title",
    "price"
).show(20, truncate=False)

+----------+----------------+-----+
|asin      |title           |price|
+----------+----------------+-----+
|B008773WWQ|Raptr [Download]|$0.00|
+----------+----------------+-----+



In [39]:
# Pake yang clean
metadata = metadata.drop(
    "price"
)

In [40]:
metadata.show(5)

[Stage 101:>                                                        (0 + 4) / 4]

+------------+--------------------+----------+------------+--------------------+--------------------+-------+--------------------+----------------+--------------------+--------------------+-----------+
|    also_buy|           also_view|      asin|       brand|            category|         description|details|             feature|        main_cat|                rank|               title|price_clean|
+------------+--------------------+----------+------------+--------------------+--------------------+-------+--------------------+----------------+--------------------+--------------------+-----------+
|          []|[B00005V8TB, B008...|0439542804|        LEGO|[Video Games, PC,...|[There is so much...|   NULL|[The action adven...|Toys &amp; Games|[">#930,608 in To...|  LEGO Island 2 - PC|       NULL|
|[B079X6BSYT]|                  []|0977070670|     Unknown|                  []|[One day, a beaut...|   NULL|                  []|     Video Games|[">#96,708 in Vid...|Ai Yori Aoshi - I...|   

In [41]:
reviews.columns

['asin',
 'overall',
 'reviewText',
 'reviewerID',
 'summary',
 'verified',
 'vote',
 'review_datetime']

In [42]:
metadata.columns

['also_buy',
 'also_view',
 'asin',
 'brand',
 'category',
 'description',
 'details',
 'feature',
 'main_cat',
 'rank',
 'title',
 'price_clean']

Handle Buat nanti Join

In [43]:
reviews.count()

2491137

Di EDA kita dapat bahwa ada reviews yang tidak punya matching asin dengan metadatanya sehingga tidak ada kontribusi kalau disimpan sehingga dihapus berikut adalah pengingat

In [44]:
missing_metadata_asin = (
    reviews
    .select("asin")
    .distinct()
    .join(
        metadata.select("asin").distinct(),
        on="asin",
        how="left_anti"
    )
)

missing_count = missing_metadata_asin.count()

print("ASIN in reviews without metadata:", missing_count)

[Stage 113:==================================================>    (12 + 1) / 13]

ASIN in reviews without metadata: 73


In [45]:
missing_reviews = (
    reviews
    .join(
        missing_metadata_asin,
        on="asin",
        how="inner"
    )
)

In [46]:
missing_review_count = missing_reviews.count()

print(
    "Reviews without metadata:",
    missing_review_count
)

Reviews without metadata: 2225


In [47]:
reviews_clean = (
    reviews
    .join(
        metadata.select("asin"),
        on="asin",
        how="inner"
    )
)

In [48]:
print(
    "Reviews after removing unmatched ASIN:",
    reviews_clean.count()
)

[Stage 140:============================>                           (8 + 4) / 16]

Reviews after removing unmatched ASIN: 2488912


Terus ada juga metadata yang tidak punya review nya jadi kayak katalog yang ga direview jadi delete juga

In [49]:
unused_metadata = (
    metadata
    .select("asin")
    .distinct()
    .join(
        reviews.select("asin").distinct(),
        on="asin",
        how="left_anti"
    )
)

print(
    "Metadata products without reviews:",
    unused_metadata.count()
)

[Stage 144:=============> (12 + 1) / 13][Stage 145:============>    (3 + 1) / 4]

Metadata products without reviews: 2


Tapi nanti di join juga ilang si jadi tunggu nanti aja dah

Gas sekarang kita join

In [50]:
print("Reviews before join:", reviews_clean.count())
print("Metadata records:", metadata.count())

Reviews before join: 2488912


[Stage 162:============================>                            (2 + 2) / 4]

Metadata records: 71911


In [51]:
final_df = reviews_clean.join(
    metadata,
    on="asin",
    how="inner"
)

In [52]:
print("Rows:", final_df.count())

[Stage 173:============================================>          (13 + 3) / 16]

Rows: 2488912


In [55]:
final_df = final_df.coalesce(10)

In [53]:
final_df.groupBy("asin") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(10)

[Stage 182:=========================================>             (12 + 4) / 16]

+----------+-----+
|      asin|count|
+----------+-----+
|B00HTK1NCS| 6462|
|B004RMK57U| 5135|
|B00KKAQYXM| 4359|
|B00JJNQG98| 3962|
|B003ZSP0WW| 3960|
|B00178630A| 3820|
|B00E4MQODC| 3520|
|B000B9RI14| 3345|
|B0009VXBAQ| 3167|
|B00DBDPOZ4| 3162|
+----------+-----+
only showing top 10 rows


In [58]:
duplicate_check = (
    final_df
    .groupBy(
        "asin",
        "reviewerID",
        "review_datetime"
    )
    .count()
    .filter("count > 1")
    .orderBy("count", ascending=False)
)

duplicate_check.show(10)

[Stage 191:=================================>                      (6 + 4) / 10]

+----------+--------------+-------------------+-----+
|      asin|    reviewerID|    review_datetime|count|
+----------+--------------+-------------------+-----+
|B00SN1QEGW|A14KKV5U5XSHXY|2018-08-02 00:00:00|   70|
|B00SN1QEGW| A71Z5AIGEFK11|2018-08-01 00:00:00|   29|
|B00SN1QEGW| A71Z5AIGEFK11|2018-07-31 00:00:00|   19|
|B014P7QI6I| A8J2YHESWGF2E|2018-08-13 00:00:00|    6|
|B014P7QI6I|A2HYAYOI8XX1IT|2018-08-13 00:00:00|    6|
|B014P7QI6I| AZQGB17PI6D3N|2018-08-13 00:00:00|    6|
|B00KN7Y3EY|A2ERJQ6T0RCLDG|2015-07-04 00:00:00|    5|
|B00004R9I4| A8TCAEBMUQ3VP|2000-09-25 00:00:00|    4|
|B00EHJIWMI|A3SOP8VC440QKC|2014-08-17 00:00:00|    4|
|B0084FGC6Y| ANUDFZ12UCKI1|2013-05-28 00:00:00|    4|
+----------+--------------+-------------------+-----+
only showing top 10 rows


In [59]:
duplicate_check.count()

883

In [61]:
duplicate_asin = (
    duplicate_check
    .select("asin")
    .limit(10)
)

In [64]:
final_df.filter(
    (final_df.asin == "B00SN1QEGW") &
    (final_df.reviewerID == "A14KKV5U5XSHXY") &
    (final_df.review_datetime.startswith("2018-08-02"))
).select(
    "asin",
    "reviewerID",
    "overall",
    "reviewText",
    "summary"
).show(
    truncate=False
)

+----------+--------------+-------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------+
|asin      |reviewerID    |overall|reviewText                                                                                                                                                                                                                                                                                                                                                         |summary                                                    |
+----------+--------------+-------+-------------------------------------------------------------

In [65]:
final_df.filter(
    (final_df.asin == "B00SN1QEGW") &
    (final_df.reviewerID == "A14KKV5U5XSHXY")
).select(
    "summary",
    "reviewText"
).count()

70

In [66]:
from pyspark.sql.functions import countDistinct

final_df.filter(
    (final_df.asin == "B00SN1QEGW") &
    (final_df.reviewerID == "A14KKV5U5XSHXY")
).agg(
    countDistinct("reviewText").alias("unique_reviews")
).show()

+--------------+
|unique_reviews|
+--------------+
|            70|
+--------------+



In [68]:
review_duplicate_check = (
    reviews
    .groupBy(
        "asin",
        "reviewerID",
        "review_datetime"
    )
    .count()
    .filter("count > 1")
    .orderBy("count", ascending=False)
)

review_duplicate_check.show(20, truncate=False)

[Stage 240:=========================================>             (12 + 4) / 16]

+----------+--------------+-------------------+-----+
|asin      |reviewerID    |review_datetime    |count|
+----------+--------------+-------------------+-----+
|B00SN1QEGW|A14KKV5U5XSHXY|2018-08-02 00:00:00|70   |
|B00SN1QEGW|A71Z5AIGEFK11 |2018-08-01 00:00:00|29   |
|B00SN1QEGW|A71Z5AIGEFK11 |2018-07-31 00:00:00|19   |
|B014P7QI6I|A2HYAYOI8XX1IT|2018-08-13 00:00:00|6    |
|B014P7QI6I|A8J2YHESWGF2E |2018-08-13 00:00:00|6    |
|B014P7QI6I|AZQGB17PI6D3N |2018-08-13 00:00:00|6    |
|B00KN7Y3EY|A2ERJQ6T0RCLDG|2015-07-04 00:00:00|5    |
|B00EHJIWMI|A3SOP8VC440QKC|2014-08-17 00:00:00|4    |
|B00004R9I4|A8TCAEBMUQ3VP |2000-09-25 00:00:00|4    |
|B0084FGC6Y|ANUDFZ12UCKI1 |2013-05-28 00:00:00|4    |
|B00OFPES1I|A3OHCEA601UIQX|2016-04-08 00:00:00|3    |
|B0156FKV5M|A32LRPM3KLP8WM|2016-04-18 00:00:00|3    |
|B00005NZ1G|A11VW4GRPBMF3A|2002-08-29 00:00:00|3    |
|B00BGA9X9W|A1VAUSR1UH5U4V|2016-12-26 00:00:00|3    |
|B001AYNH3W|A19BN36NSWNP37|2008-09-08 00:00:00|3    |
|B015YE7HBQ|A2LLVLJ84Q8JUH|2

In [69]:
print(
    "Duplicate groups:",
    review_duplicate_check.count()
)

[Stage 249:======================>                                  (2 + 3) / 5]

Duplicate groups: 883


In [70]:
duplicate_rows = (
    review_duplicate_check
    .selectExpr("sum(count) as total_duplicate_rows")
)

duplicate_rows.show()

[Stage 259:======================>                                  (2 + 3) / 5]

+--------------------+
|total_duplicate_rows|
+--------------------+
|                1927|
+--------------------+



In [ ]:
(
    reviews
    .filter(
        (reviews.asin == "B00SN1QEGW") &
        (reviews.reviewerID == "A14KKV5U5XSHXY")
    )
    .select(
        "asin",
        "reviewerID",
        "style",
        "summary"
    )
    .show(100, truncate=False)
)

In [ ]:
reviews.filter(
    reviews.reviewerID=="A14KKV5U5XSHXY"
).select(
    "asin",
    "reviewerID",
    "reviewTime",
    "overall",
    "summary"
).count()

In [ ]:
reviews.filter(
    reviews.reviewerID=="A14KKV5U5XSHXY"
).groupBy(
    "asin"
).count().orderBy(
    "count",
    ascending=False
).show()

In [ ]:
reviews.filter(
    F.col("reviewerID")=="A14KKV5U5XSHXY"
).select(
    "summary",
    "reviewText"
).distinct().count()

In [ ]:
reviews.filter(
    F.col("reviewerID")=="A14KKV5U5XSHXY"
).count()

In [ ]:
reviews.groupBy("reviewerID") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(20)

In [ ]:
reviews.filter(
    F.col("reviewerID") == "AJKWF4W7QD4NS"
).groupBy("asin") \
 .count() \
 .orderBy(F.desc("count")) \
 .show(20, truncate=False)

In [ ]:
duplicates = (
    reviews
    .groupBy("reviewerID", "asin")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

duplicates.show(20, truncate=False)

In [ ]:
reviews.groupBy("reviewerID", "asin") \
    .count() \
    .filter(F.col("count") > 1) \
    .groupBy("count") \
    .count() \
    .orderBy("count") \
    .show()

In [73]:
final_df.write \
    .mode("overwrite") \
    .parquet("hdfs://namenode:9000/data/amazon_reviews/final_reviews_parquet")


In [74]:
spark

In [76]:
path = "hdfs://namenode:9000/data/amazon_reviews/final_reviews_parquet"

parquet_df = spark.read.parquet(path)

parquet_df.show(5, truncate=False)

[Stage 277:>                                                        (0 + 1) / 1]

+----------+-------+----------------------------------------------------------------------------------------------------------------------------------------------+--------------+-------------+--------+----+-------------------+--------+------------------------------------------------------------------------+---------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [77]:
original_count = final_df.count()
saved_count = parquet_df.count()

print("Original:", original_count)
print("Saved:", saved_count)
print("Complete:", original_count == saved_count)

Original: 2488912
Saved: 2488912
Complete: True


In [78]:
HDFS = "hdfs://namenode:9000"
df = spark.read.parquet(f"{HDFS}/data/amazon_reviews/final_reviews_parquet")

print("Baris:", df.count())      # harusnya 2.488.912 sesuai Bab 4.3.2
df.printSchema()

# cek null per kolom
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

# cek rating dan duplikat
df.groupBy("overall").count().orderBy("overall").show()
print("Duplikat:", df.count() - df.dropDuplicates().count())

Baris: 2488912
root
 |-- asin: string (nullable = true)
 |-- overall: double (nullable = true)
 |-- reviewText: string (nullable = true)
 |-- reviewerID: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- verified: boolean (nullable = true)
 |-- vote: integer (nullable = true)
 |-- review_datetime: string (nullable = true)
 |-- also_buy: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- also_view: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- brand: string (nullable = true)
 |-- category: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- description: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- details: struct (nullable = true)
 |    |-- \n    Item Weight: \n    : string (nullable = true)
 |    |-- \n    Package Dimensions: \n    : string (nullable = true)
 |    |-- \n    Product Dimensions: \n    : string (nullable = true)
 |    |--  Date first listed 

+----+-------+----------+----------+-------+--------+----+---------------+--------+---------+-----+--------+-----------+-------+-------+--------+----+-----+-----------+
|asin|overall|reviewText|reviewerID|summary|verified|vote|review_datetime|also_buy|also_view|brand|category|description|details|feature|main_cat|rank|title|price_clean|
+----+-------+----------+----------+-------+--------+----+---------------+--------+---------+-----+--------+-----------+-------+-------+--------+----+-----+-----------+
|   0|      0|         0|         0|    797|       0|   0|              0|       0|        0|    0|       0|          0|   1200|      0|       0|   0|    0|    1998653|
+----+-------+----------+----------+-------+--------+----+---------------+--------+---------+-----+--------+-----------+-------+-------+--------+----+-----+-----------+

+-------+-------+
|overall|  count|
+-------+-------+
|    1.0| 302060|
|    2.0| 136867|
|    3.0| 205509|
|    4.0| 397634|
|    5.0|1446842|
+-------+-

[Stage 305:===================================================>   (15 + 1) / 16]

Duplikat: 151


In [79]:
df = spark.read.parquet("hdfs://namenode:9000/data/amazon_reviews/final_reviews_parquet")
df.select((F.col("vote") > 0).alias("helpful")).groupBy("helpful").count().show()
df.groupBy(F.year("review_datetime").alias("tahun")).agg(
    F.avg((F.col("vote") > 0).cast("int")).alias("rasio_helpful")).orderBy("tahun").show()

+-------+-------+
|helpful|  count|
+-------+-------+
|   true| 422999|
|  false|2065913|
+-------+-------+



[Stage 313:============================================>           (8 + 2) / 10]

+-----+-------------------+
|tahun|      rasio_helpful|
+-----+-------------------+
| 1997|                0.2|
| 1998|               0.25|
| 1999| 0.8524064171122995|
| 2000| 0.6517068273092369|
| 2001| 0.4908696595098677|
| 2002|0.44206053795912637|
| 2003| 0.4771615327410075|
| 2004| 0.5991169977924945|
| 2005| 0.6742115173387729|
| 2006| 0.6707983364592677|
| 2007| 0.4887945434298441|
| 2008|0.41547313666809177|
| 2009| 0.3556746967170701|
| 2010|0.33861447198402445|
| 2011| 0.2980764635603345|
| 2012|0.24131246103524548|
| 2013|0.15348036888858752|
| 2014|0.11858480920426367|
| 2015|0.09234717093403805|
| 2016|0.10150275941767234|
+-----+-------------------+
only showing top 20 rows
